# Short Rate Forecasting with Vasicek and CIR Models using LSTM

This notebook explores **interest rate modeling** using two classical stochastic short-rate models — the **Vasicek** and **Cox-Ingersoll-Ross (CIR)** models — and applies an **LSTM neural network** to forecast future short rates.

Project steps:
1. Simulate short-rate paths using Vasicek and CIR models.
2. Prepare the simulated data for ML forecasting.
3. Train an LSTM network to predict future rates.
4. Evaluate model performance using MAE, RMSE, and MAPE.
5. Visualize predictions and compare with true rates.

In [ ]:
# Importing Libraries
import sys
sys.path.append("../src")

import numpy as np
import matplotlib.pyplot as plt

from rate_models import vasicek_sim, cir_sim
from lstm_forecast import train_forecast_lstm
from evaluate import evaluate_forecast

np.random.seed(42)

## Vasicek Model

The **Vasicek model** is a mean-reverting model for interest rates:

$$ dr_t = a(b - r_t)dt + \sigma dW_t $$

Where:
- $r_t$: short rate at time $t$
- $a$: speed of mean reversion
- $b$: long-term mean interest rate
- $\sigma$: volatility
- $W_t$: Wiener process (Brownian motion)

**Pros:** simple and analytically tractable.
**Cons:** allows negative interest rates (not always realistic).

In [ ]:
r0, a, b, sigma = 0.03, 0.10, 0.05, 0.02
T, dt = 5.0, 0.01

rates_vasicek = vasicek_sim(r0, a, b, sigma, T, dt, seed=42)

plt.figure(figsize=(10, 4))
plt.plot(rates_vasicek)
plt.title("Simulated Vasicek Short Rate Path")
plt.xlabel("Time step")
plt.ylabel("Short rate")
plt.show()

## Cox-Ingersoll-Ross (CIR) Model

The **CIR model** modifies Vasicek to ensure rates remain positive:

$$ dr_t = a(b - r_t)dt + \sigma \sqrt{r_t}\, dW_t $$

This ensures $r_t \geq 0$ as long as the **Feller condition** holds:

$$ 2ab \geq \sigma^2 $$

**Pros:** keeps interest rates non-negative.
**Cons:** slightly more complex than Vasicek.

In [ ]:
rates_cir = cir_sim(r0, a, b, sigma, T, dt, seed=42)

plt.figure(figsize=(10, 4))
plt.plot(rates_cir, color="darkorange")
plt.title("Simulated CIR Short Rate Path")
plt.xlabel("Time step")
plt.ylabel("Short rate")
plt.show()

## LSTM Forecasting

**Why LSTM?** LSTMs (Long Short-Term Memory networks) are a type of recurrent neural network designed to capture long-term dependencies in sequential data, making them well suited to time-series forecasting.

**Approach:**
1. Transform simulated rates into a supervised-learning format with a lag window.
2. Train an LSTM to predict one-step-ahead rates.
3. Evaluate predictions using MAE, RMSE, and MAPE.

In [ ]:
def run_pipeline(name, series, n_lags=10, epochs=20, batch_size=16):
    split_idx = int(len(series) * 0.8)
    train_series = series[:split_idx]
    test_series = series[split_idx - n_lags:]  # include lag window for test prep

    y_true, y_pred, _ = train_forecast_lstm(
        train_series, test_series, n_lags=n_lags, epochs=epochs, batch_size=batch_size
    )
    metrics = evaluate_forecast(y_true, y_pred)
    print(f"{name}: MAE={metrics['MAE']:.5f}  RMSE={metrics['RMSE']:.5f}  MAPE={metrics['MAPE']:.2f}%")

    plt.figure(figsize=(10, 4))
    plt.plot(y_true, label="True rate")
    plt.plot(y_pred, label="LSTM forecast")
    plt.title(f"{name}: True vs Forecasted Short Rate")
    plt.xlabel("Time step")
    plt.ylabel("Short rate")
    plt.legend()
    plt.show()

    return metrics

vasicek_metrics = run_pipeline("Vasicek", rates_vasicek)
cir_metrics = run_pipeline("CIR", rates_cir)

## Results Interpretation

- **MAE** tells us the average absolute forecast error size.
- **RMSE** penalizes large errors more than MAE.
- **MAPE** expresses forecast error in percentage terms.

We trained the LSTM separately on Vasicek- and CIR-simulated data and compared error metrics for both. Lower error metrics indicate a better forecast; in our runs, the CIR-simulated series (which stays away from zero given the chosen parameters) tends to yield lower relative error than the Vasicek series.

**Note:** Since this uses simulated data, real-world performance on historical rate data may differ, and exact metric values will vary run-to-run due to random initialization and simulation noise.